# Food-101: v4 - Transfer Learning con EfficientNet-B4
## TensorFlow 2.10.1 Compatible

## Estrategia
En lugar de entrenar desde cero, usamos **EfficientNet-B4 preentrenado** con fine-tuning progresivo:

1. **Fase 1:** Entrenar solo clasificador (2 epochs) - EfficientNet congelado
2. **Fase 2:** Fine-tune últimas 50 capas (25-30 epochs) - Descongelar gradualmente
3. **Fase 3:** Fine-tune completo (20 epochs) - Modelo entero descongelado


## Imports

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import warnings
import os
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU disponible: {tf.test.is_built_with_cuda()}')

## Configuración

In [ ]:
# Hiperparámetros
IMG_SIZE = 380  # Óptimo para EfficientNet-B4
BATCH_SIZE = 32
NUM_CLASSES = 15
AUTOTUNE = tf.data.AUTOTUNE

print(f'IMG_SIZE: {IMG_SIZE} | BATCH_SIZE: {BATCH_SIZE} | CLASES: {NUM_CLASSES}')
print(f'EfficientNet-B4: 19M params (ImageNet pretraining)')

## Cargar Dataset

In [ ]:
# Cargar Food-101
(train_ds, val_ds), info = tfds.load(
    'food101',
    split=['train', 'validation'],
    with_info=True,
    as_supervised=True
)

class_names = info.features['label'].names
print(f'Dataset total: {len(class_names)} clases')

# Filtrar a 15 clases
train_ds = train_ds.filter(lambda img, label: label < NUM_CLASSES)
val_ds = val_ds.filter(lambda img, label: label < NUM_CLASSES)
class_names = class_names[:NUM_CLASSES]

print(f'Usando {NUM_CLASSES} clases: {class_names[:5]}...')

# Contar samples
train_count = sum(1 for _ in train_ds)
val_count = sum(1 for _ in val_ds)
print(f'Training samples: {train_count} | Validation samples: {val_count}')

## RandAugment Implementation

In [ ]:
# Versión compatible con TF 2.10.1
def create_randaugment_layer(num_ops=2, magnitude=9):
    """Crear sequential augmentation con RandAugment"""
    augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.15),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomContrast(magnitude / 100),
        tf.keras.layers.RandomBrightness(magnitude / 100),
    ])
    return augmentation

print('RandAugment layer created (TF 2.10.1 compatible)')

In [ ]:
# Preprocessing: Resize + Normalize ImageNet
def preprocess(image, label):
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    # Normalizar a [-1, 1] (ImageNet normalization)
    image = (image / 127.5) - 1.0
    return image, label

# RandAugment
augmentation = create_randaugment_layer(num_ops=2, magnitude=9)

def augment(image, label):
    image = augmentation(image, training=True)
    return image, label

# Training dataset
train_dataset = (
    train_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(1000)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Validation dataset (sin augmentation)
val_dataset = (
    val_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f'Data pipeline configurado:')
print(f'  - Input size: {IMG_SIZE}×{IMG_SIZE}')
print(f'  - Batch size: {BATCH_SIZE}')
print(f'  - Augmentation: RandAugment (Flip + Rotation + Zoom + Contrast + Brightness)')

## Modelo: EfficientNet-B4 + Custom Classifier

In [ ]:
# Descargar EfficientNet-B4 preentrenado
print('Descargando EfficientNet-B4 (preentrenado con ImageNet)...')

base_model = tf.keras.applications.EfficientNetB4(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,  # Sin clasificador, solo features
    weights='imagenet'   # Preentrenado
)

print(f'EfficientNet-B4 descargado')
print(f'  - Parámetros: {base_model.count_params():,}')
print(f'  - Capas: {len(base_model.layers)}')

In [ ]:
# Construir modelo completo: EfficientNet + Custom classifier
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

total_params = model.count_params()
print(f'\nModelo completo:')
print(f'  - Total params: {total_params:,}')
print(f'  - EfficientNet-B4: {base_model.count_params():,} (preentrenado)')
print(f'  - Clasificador: {total_params - base_model.count_params():,} (nuevas capas)')

## FASE 1: Entrenar Clasificador (EfficientNet Congelado)

In [ ]:
# Congelar EfficientNet-B4
base_model.trainable = False

print('Fase 1: Entrenar solo clasificador')
print(f'  - EfficientNet trainable: {base_model.trainable}')
print(f'  - Clasificador trainable: {model.layers[-3].trainable}')

# Compilar - Compatible con TF 2.10.1
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# Callbacks - Compatible con TF 2.10.1
checkpoint_phase1 = tf.keras.callbacks.ModelCheckpoint(
    'best_model_v4_phase1_tf2101.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=0
)

print(f'\nEntrenando Fase 1 (2 epochs)...')

# Entrenar solo 2 epochs para calentar
history_phase1 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=2,
    callbacks=[checkpoint_phase1],
    verbose=1
)

print(f'Fase 1 completada')
print(f'  - Train accuracy: {history_phase1.history["accuracy"][-1]:.4f}')
print(f'  - Val accuracy: {history_phase1.history["val_accuracy"][-1]:.4f}')

## FASE 2: Fine-tune Capas Superiores de EfficientNet

In [ ]:
# Descongelar las últimas 50 capas del EfficientNet
num_layers_to_unfreeze = 50
for layer in base_model.layers[-num_layers_to_unfreeze:]:
    layer.trainable = True

print(f'Fase 2: Fine-tune capas superiores')
print(f'  - Capas descongeladas: últimas {num_layers_to_unfreeze}')
print(f'  - Total capas trainables: {sum(1 for layer in model.layers if layer.trainable)}')

# Recompilar con learning rate más bajo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# Callback para Phase 2
checkpoint_phase2 = tf.keras.callbacks.ModelCheckpoint(
    'best_model_v4_phase2_tf2101.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=0
)

# ReduceLROnPlateau para ajuste dinámico de learning rate
reduce_lr_phase2 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

print(f'\nEntrenando Fase 2 (25 epochs)...')

history_phase2 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=25,
    callbacks=[checkpoint_phase2, reduce_lr_phase2],
    verbose=1
)

print(f'Fase 2 completada')
print(f'  - Train accuracy: {history_phase2.history["accuracy"][-1]:.4f}')
print(f'  - Val accuracy: {history_phase2.history["val_accuracy"][-1]:.4f}')

## FASE 3: Fine-tune Completo

In [ ]:
# Descongelar TODO
base_model.trainable = True

print(f'Fase 3: Fine-tune completo')
print(f'  - EfficientNet trainable: {base_model.trainable}')
print(f'  - Total capas trainables: {sum(1 for layer in model.layers if layer.trainable)}')

# Recompilar con learning rate aún más bajo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# Callback para Phase 3
checkpoint_phase3 = tf.keras.callbacks.ModelCheckpoint(
    'best_model_v4_phase3_tf2101.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=0
)

# ReduceLROnPlateau
reduce_lr_phase3 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-8,
    verbose=1
)

print(f'\nEntrenando Fase 3 (20 epochs)...')

history_phase3 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=[checkpoint_phase3, reduce_lr_phase3],
    verbose=1
)

print(f'Fase 3 completada')
print(f'  - Train accuracy: {history_phase3.history["accuracy"][-1]:.4f}')
print(f'  - Val accuracy: {history_phase3.history["val_accuracy"][-1]:.4f}')

## Evaluación Final

In [ ]:
# Cargar mejor modelo
if os.path.exists('best_model_v4_phase3_tf2101.h5'):
    model = tf.keras.models.load_model('best_model_v4_phase3_tf2101.h5')
    print("Modelo cargado desde checkpoint de Phase 3")
else:
    print("ADVERTENCIA: No se encontró best_model_v4_phase3_tf2101.h5")
    print("Asegúrate de que las fases 1, 2 y 3 se hayan entrenado completamente.")
    print("El modelo actual se usará para evaluación.")

# Evaluar
test_loss, test_acc = model.evaluate(val_dataset, verbose=0)

print(f'\n{"="*70}')
print(f'EVALUACIÓN FINAL - TRANSFER LEARNING (EfficientNet-B4)')
print(f'TensorFlow 2.10.1 Compatible')
print(f'{"="*70}')
print(f'\nValidation Loss: {test_loss:.4f}')
print(f'Validation Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'\nResumen de Fases:')
print(f'  Fase 1 (2 epochs):      Val Acc = {history_phase1.history["val_accuracy"][-1]:.4f}')
print(f'  Fase 2 ({len(history_phase2.history["val_accuracy"])} epochs):   Val Acc = {history_phase2.history["val_accuracy"][-1]:.4f}')
print(f'  Fase 3 ({len(history_phase3.history["val_accuracy"])} epochs):    Val Acc = {history_phase3.history["val_accuracy"][-1]:.4f} (FINAL)')
print(f'{"="*70}')

## Gráficas de Entrenamiento

In [ ]:
# Concatenar historiales
all_train_acc = (history_phase1.history['accuracy'] + 
                 history_phase2.history['accuracy'] + 
                 history_phase3.history['accuracy'])
all_val_acc = (history_phase1.history['val_accuracy'] + 
               history_phase2.history['val_accuracy'] + 
               history_phase3.history['val_accuracy'])

all_train_loss = (history_phase1.history['loss'] + 
                  history_phase2.history['loss'] + 
                  history_phase3.history['loss'])
all_val_loss = (history_phase1.history['val_loss'] + 
                history_phase2.history['val_loss'] + 
                history_phase3.history['val_loss'])

# Puntos donde cambian fases
phase1_end = len(history_phase1.history['accuracy'])
phase2_end = phase1_end + len(history_phase2.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(all_train_acc, label='Training', linewidth=2)
ax1.plot(all_val_acc, label='Validation', linewidth=2)
ax1.axvline(phase1_end, color='red', linestyle='--', alpha=0.5, label='Phase 2 start')
ax1.axvline(phase2_end, color='green', linestyle='--', alpha=0.5, label='Phase 3 start')
ax1.set_title('Accuracy - Transfer Learning (3 Fases)', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(all_train_loss, label='Training', linewidth=2)
ax2.plot(all_val_loss, label='Validation', linewidth=2)
ax2.axvline(phase1_end, color='red', linestyle='--', alpha=0.5, label='Phase 2 start')
ax2.axvline(phase2_end, color='green', linestyle='--', alpha=0.5, label='Phase 3 start')
ax2.set_title('Loss - Transfer Learning (3 Fases)', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Análisis Detallado

In [ ]:
# Obtener predicciones
y_true = []
y_pred = []
y_pred_proba = []

for images, labels in val_dataset:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))
    y_pred_proba.extend(predictions)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_pred_proba = np.array(y_pred_proba)

print(f'Predicciones obtenidas: {len(y_true)}')

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)

# Visualizar
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

ax.set(xticks=np.arange(cm_norm.shape[1]),
        yticks=np.arange(cm_norm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
ax.set_title('Matriz de Confusión Normalizada - Transfer Learning', fontsize=12)
ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy por clase
class_accuracy = cm.diagonal() / cm.sum(axis=1)
class_accuracy_df = pd.DataFrame({
    'Class': class_names,
    'Accuracy': class_accuracy,
    'Correct': cm.diagonal(),
    'Total': cm.sum(axis=1)
}).sort_values('Accuracy', ascending=False)

print('\n=== ACCURACY POR CLASE ===')
print(class_accuracy_df.to_string(index=False))
print(f'\nMejor: {class_accuracy_df.iloc[0]["Class"]} ({class_accuracy_df.iloc[0]["Accuracy"]:.2%})')
print(f'Peor: {class_accuracy_df.iloc[-1]["Class"]} ({class_accuracy_df.iloc[-1]["Accuracy"]:.2%})')

In [ ]:
# Top-10 confusiones
confusion_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'True': class_names[i],
                'Predicted': class_names[j],
                'Count': cm[i, j],
                'Percentage': f'{100 * cm[i, j] / cm[i].sum():.1f}%'
            })

confusion_df = pd.DataFrame(confusion_pairs).sort_values('Count', ascending=False)

print('\n=== TOP-10 CONFUSIONES ENTRE CLASES ===')
print(confusion_df.head(10).to_string(index=False))

In [ ]:
# Classification report
print('\n=== CLASSIFICATION REPORT ===')
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

## Conclusiones y Comparación

In [ ]:
print(f'\n{"="*70}')
print(f'RESULTADOS FINALES: TRANSFER LEARNING')
print(f'TensorFlow 2.10.1')
print(f'{"="*70}')
print(f'\n Accuracy: {test_acc*100:.2f}%')
print(f'\n  Arquitectura:')
print(f'   - Base: EfficientNet-B4 (19M params, ImageNet pretraining)')
print(f'   - Clasificador: 512 → 256 → {NUM_CLASSES} clases')
print(f'   - Total params: {total_params:,}')
print(f'\n  Training Strategy:')
print(f'   - Fase 1: Congelar backbone (2 epochs)')
print(f'   - Fase 2: Fine-tune últimas 50 capas ({len(history_phase2.history["val_accuracy"])} epochs)')
print(f'   - Fase 3: Fine-tune completo ({len(history_phase3.history["val_accuracy"])} epochs)')
print(f'\n  Técnicas utilizadas:')
print(f'   - RandAugment: Flip + Rotation + Zoom + Contrast + Brightness')
print(f'   - ReduceLROnPlateau callback')
print(f'   - Progressive learning rate: 0.001 → 0.0001 → 0.00001')
print(f'\n  Modelos guardados:')
print(f'   - best_model_v4_phase1_tf2101.h5')
print(f'   - best_model_v4_phase2_tf2101.h5')
print(f'   - best_model_v4_phase3_tf2101.h5 (MEJOR - {test_acc*100:.2f}%)')
print(f'{"="*70}')